# Descriptive and Statistical Analysis: Genetic Algorithm (GA) vs Random Search (RS)

This notebook uses `ga_rs_fold_results.csv`, performs analyses separately for each model, and excludes Decision Tree. Models analyzed: Random Forest, XGBoost, Logistic Regression, and SVM (based on the names available in the CSV).

**Important:** GA and Random Search results are paired using `Model`, `Repeat`, `Outer_Fold`, and `Split_ID`. Verify that these splits are identical across both optimizers. Repeated cross-validation also means that scores across splits are not fully independent; interpret p-values cautiously and describe the validation design in the paper.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
from IPython.display import display

# Change the input filename if necessary
INPUT_FILE = Path("ga_rs_fold_results.csv")
OUTPUT_DIR = Path("ga_rs_analysis_output")
OUTPUT_DIR.mkdir(exist_ok=True)
ALPHA = 0.05

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"File {INPUT_FILE} was not found. Place the file in the notebook working directory or update INPUT_FILE."
    )

df = pd.read_csv(INPUT_FILE)
print("Initial data shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Models found:", sorted(df["Model"].dropna().astype(str).unique()))
print("Optimizers found:", sorted(df["Optimizer"].dropna().astype(str).unique()))

display(df.head())

Ukuran data awal: (200, 28)
Kolom: ['Model', 'Optimizer', 'Repeat', 'Outer_Fold', 'Split_ID', 'Inner_CV_Folds', 'Search_Scoring', 'Best_Inner_CV_MCC', 'Tuning_And_Refit_Seconds', 'Actual_Candidate_Evaluations', 'Unique_Candidate_Configurations', 'Planned_Budget', 'Target_Prevalence_Test', 'accuracy', 'precision', 'recall_sensitivity', 'specificity', 'f1', 'roc_auc', 'average_precision', 'pr_auc', 'balanced_accuracy', 'mcc', 'TN', 'FP', 'FN', 'TP', 'Source_File']
Model ditemukan: ['Decision Tree', 'Logistic Regression', 'Random Forest', 'SVM', 'XGBoost']
Optimizer ditemukan: ['GA', 'RandomSearch']


,Model,Optimizer,Repeat,Outer_Fold,Split_ID,Inner_CV_Folds,Search_Scoring,Best_Inner_CV_MCC,Tuning_And_Refit_Seconds,Actual_Candidate_Evaluations,...,roc_auc,average_precision,pr_auc,balanced_accuracy,mcc,TN,FP,FN,TP,Source_File
0,Random Forest,GA,1,1,1,3,MCC,0.277512,485.794322,210,...,0.863868,0.194717,0.183901,0.809465,0.327013,398,88,5,20,ga_rs_fold_results 2.csv
1,Random Forest,RandomSearch,1,1,1,3,MCC,0.268421,68.817878,100,...,0.858272,0.201196,0.189385,0.816091,0.319871,385,101,4,21,ga_rs_fold_results 2.csv
2,Random Forest,GA,1,2,2,3,MCC,0.276366,468.939168,210,...,0.783868,0.226823,0.215602,0.767984,0.307598,416,70,8,17,ga_rs_fold_results 2.csv
3,Random Forest,RandomSearch,1,2,2,3,MCC,0.266691,79.661792,100,...,0.785021,0.213250,0.191811,0.751070,0.293665,419,67,9,16,ga_rs_fold_results 2.csv
4,Random Forest,GA,1,3,3,3,MCC,0.284754,403.129321,210,...,0.869712,0.229124,0.207279,0.792428,0.279842,362,124,4,21,ga_rs_fold_results 2.csv


In [2]:
# ------------------------------------------------------------
# 1. Validate and clean split identifiers
# ------------------------------------------------------------

required_keys = ["Model", "Optimizer", "Repeat", "Outer_Fold", "Split_ID"]
missing_keys = [c for c in required_keys if c not in df.columns]
if missing_keys:
    raise ValueError(f"Required identifier columns are missing: {missing_keys}")

# Exclude Decision Tree. Other model names follow the CSV.
df["Model"] = df["Model"].astype(str).str.strip()
df["Optimizer"] = df["Optimizer"].astype(str).str.strip()
model_exclude = df["Model"].str.contains(r"decision\s*tree|^dt$", case=False, regex=True)
df = df.loc[~model_exclude].copy()

# Normalisasi nama optimizer ke GA dan Random Search
opt_norm = {
    "ga": "GA",
    "genetic algorithm": "GA",
    "geneticalgorithm": "GA",
    "random search": "Random Search",
    "randomsearch": "Random Search",
    "rs": "Random Search",
}
df["Optimizer"] = df["Optimizer"].apply(
    lambda x: opt_norm.get(x.strip().lower(), x)
)

expected_optimizers = {"GA", "Random Search"}
found_optimizers = set(df["Optimizer"].dropna().unique())
if not expected_optimizers.issubset(found_optimizers):
    raise ValueError(
        f"Both GA and Random Search data are required. Found: {sorted(found_optimizers)}"
    )

# Prediction metrics to be analyzed
metrics = [
    "accuracy", "precision", "recall_sensitivity", "specificity", "f1",
    "roc_auc", "average_precision", "pr_auc", "balanced_accuracy", "mcc"
]
metrics = [m for m in metrics if m in df.columns]
if not metrics:
    raise ValueError("No recognized prediction metric columns were found.")
print("Metrics analyzed:", metrics)

# Remove duplicate experiment identifiers only when the entire rows are identical.
# If identifiers match but results differ, stop to avoid selecting an arbitrary result.
key = ["Model", "Optimizer", "Repeat", "Outer_Fold", "Split_ID"]
repeated_key = df.duplicated(key, keep=False)
if repeated_key.any():
    duplicated_rows = df.loc[repeated_key].sort_values(key)
    compare_cols = [c for c in df.columns if c != "Source_File"]
    conflicts = []
    for _, group in duplicated_rows.groupby(key, dropna=False):
        if len(group[compare_cols].drop_duplicates()) > 1:
            conflicts.append(group)
    if conflicts:
        print("Examples of experiment identifiers with conflicting values:")
        display(pd.concat(conflicts, ignore_index=True).head(20))
        raise ValueError(
            "Some Model/Optimizer/Repeat/Outer_Fold/Split_ID keys occur more than once with conflicting results. "
            "Check the source file before performing statistical analysis."
        )
    before = len(df)
    df = df.drop_duplicates(subset=key, keep="first").copy()
    print(f"Identical duplicate rows removed: {before - len(df)}")

# Check whether both optimizers are present for every model/split combination
coverage = df.groupby(["Model", "Repeat", "Outer_Fold", "Split_ID"])["Optimizer"].nunique()
print("Number of splits missing one of the optimizers:", int((coverage != 2).sum()))

# Save the cleaned data used in the analysis
clean_path = OUTPUT_DIR / "ga_rs_fold_results_clean.csv"
df.to_csv(clean_path, index=False)
print("Cleaned data saved to:", clean_path)
print("Cleaned data shape:", df.shape)
display(df.groupby(["Model", "Optimizer"]).size().rename("N_rows").reset_index())

Metrik yang dianalisis: ['accuracy', 'precision', 'recall_sensitivity', 'specificity', 'f1', 'roc_auc', 'average_precision', 'pr_auc', 'balanced_accuracy', 'mcc']
Jumlah split yang tidak memiliki kedua optimizer: 0
Data bersih disimpan ke: ga_rs_analysis_output\ga_rs_fold_results_clean.csv
Ukuran data bersih: (160, 28)


,Model,Optimizer,N_rows
0,Logistic Regression,GA,20
1,Logistic Regression,Random Search,20
2,Random Forest,GA,20
3,Random Forest,Random Search,20
4,SVM,GA,20
5,SVM,Random Search,20
6,XGBoost,GA,20
7,XGBoost,Random Search,20


In [3]:
# ------------------------------------------------------------
# 2. Descriptive analysis by model and optimizer
# ------------------------------------------------------------

# Additional search/tuning efficiency metrics (if available)
extra_descriptive = [
    "Tuning_And_Refit_Seconds",
    "Actual_Candidate_Evaluations",
    "Unique_Candidate_Configurations",
    "Best_Inner_CV_MCC",
]
all_descriptive_metrics = metrics + [c for c in extra_descriptive if c in df.columns]

long_desc = df.melt(
    id_vars=["Model", "Optimizer", "Repeat", "Outer_Fold", "Split_ID"],
    value_vars=all_descriptive_metrics,
    var_name="Metric",
    value_name="Value",
)

descriptive = (
    long_desc.groupby(["Model", "Optimizer", "Metric"], dropna=False)["Value"]
    .agg(N="count", Mean="mean", SD="std", Median="median", Min="min", Max="max")
    .reset_index()
)
for col in ["Mean", "SD", "Median", "Min", "Max"]:
    descriptive[col] = descriptive[col].round(6)

print("Descriptive statistics by model, optimizer, and metric:")
for model in sorted(descriptive["Model"].unique()):
    print("\n" + "=" * 90 + f"\nMODEL: {model}\n" + "=" * 90)
    display(descriptive[descriptive["Model"] == model].reset_index(drop=True))

desc_path = OUTPUT_DIR / "ga_rs_descriptive_statistics.csv"
descriptive.to_csv(desc_path, index=False)
print("Descriptive statistics saved to:", desc_path)

Statistik deskriptif per model, optimizer, dan metrik:

MODEL: Logistic Regression


,Model,Optimizer,Metric,N,Mean,SD,Median,Min,Max
0,Logistic Regression,GA,Actual_Candidate_Evaluations,20,210.000000,0.000000,210.000000,210.000000,210.000000
1,Logistic Regression,GA,Best_Inner_CV_MCC,20,0.257449,0.007944,0.255940,0.243645,0.272157
2,Logistic Regression,GA,Tuning_And_Refit_Seconds,20,118.820715,7.747668,120.199977,97.945158,134.185776
3,Logistic Regression,GA,Unique_Candidate_Configurations,20,201.600000,2.479389,202.000000,196.000000,206.000000
4,Logistic Regression,GA,accuracy,20,0.736204,0.013744,0.736791,0.712329,0.759295
5,Logistic Regression,GA,average_precision,20,0.212296,0.042173,0.213224,0.118619,0.272931
6,Logistic Regression,GA,balanced_accuracy,20,0.769003,0.036310,0.770622,0.700658,0.838025
7,Logistic Regression,GA,f1,20,0.229074,0.017201,0.230335,0.201183,0.256684
8,Logistic Regression,GA,mcc,20,0.254317,0.032076,0.252844,0.197774,0.313398
9,Logistic Regression,GA,pr_auc,20,0.199303,0.041982,0.198935,0.109344,0.264005



MODEL: Random Forest


,Model,Optimizer,Metric,N,Mean,SD,Median,Min,Max
0,Random Forest,GA,Actual_Candidate_Evaluations,20,210.000000,0.000000,210.000000,210.000000,210.000000
1,Random Forest,GA,Best_Inner_CV_MCC,20,0.273954,0.008538,0.273503,0.258404,0.291457
2,Random Forest,GA,Tuning_And_Refit_Seconds,20,439.690847,39.701589,445.244637,353.822601,498.193356
3,Random Forest,GA,Unique_Candidate_Configurations,20,199.450000,4.382681,198.500000,192.000000,207.000000
4,Random Forest,GA,accuracy,20,0.787280,0.040051,0.785714,0.727984,0.847358
5,Random Forest,GA,average_precision,20,0.199857,0.041165,0.186845,0.140758,0.291645
6,Random Forest,GA,balanced_accuracy,20,0.757701,0.041780,0.768767,0.674609,0.814609
7,Random Forest,GA,f1,20,0.252374,0.035818,0.250035,0.187135,0.315789
8,Random Forest,GA,mcc,20,0.264573,0.044543,0.262369,0.185860,0.338330
9,Random Forest,GA,pr_auc,20,0.186737,0.041150,0.174419,0.132899,0.278717



MODEL: SVM


,Model,Optimizer,Metric,N,Mean,SD,Median,Min,Max
0,SVM,GA,Actual_Candidate_Evaluations,20,210.000000,0.000000,210.000000,210.000000,210.000000
1,SVM,GA,Best_Inner_CV_MCC,20,0.242911,0.005535,0.242000,0.232563,0.258570
2,SVM,GA,Tuning_And_Refit_Seconds,20,289.605130,10.836786,289.820098,266.979423,313.438652
3,SVM,GA,Unique_Candidate_Configurations,20,203.250000,2.149051,204.000000,198.000000,207.000000
4,SVM,GA,accuracy,20,0.714286,0.017013,0.714286,0.663405,0.737769
5,SVM,GA,average_precision,20,0.200699,0.055533,0.190563,0.122754,0.330719
6,SVM,GA,balanced_accuracy,20,0.749768,0.035211,0.746481,0.690247,0.808765
7,SVM,GA,f1,20,0.212287,0.018382,0.212887,0.173077,0.238636
8,SVM,GA,mcc,20,0.231968,0.032519,0.229754,0.171188,0.282713
9,SVM,GA,pr_auc,20,0.187301,0.055409,0.171156,0.113385,0.317228



MODEL: XGBoost


,Model,Optimizer,Metric,N,Mean,SD,Median,Min,Max
0,XGBoost,GA,Actual_Candidate_Evaluations,20,210.000000,0.000000,210.000000,210.000000,210.000000
1,XGBoost,GA,Best_Inner_CV_MCC,20,0.271182,0.010318,0.270152,0.254716,0.295521
2,XGBoost,GA,Tuning_And_Refit_Seconds,20,80.983601,4.713809,79.792752,75.042872,91.412022
3,XGBoost,GA,Unique_Candidate_Configurations,20,207.800000,1.507874,208.000000,204.000000,210.000000
4,XGBoost,GA,accuracy,20,0.793346,0.033527,0.798434,0.726027,0.859100
5,XGBoost,GA,average_precision,20,0.211675,0.049010,0.190911,0.149279,0.284856
6,XGBoost,GA,balanced_accuracy,20,0.744424,0.042708,0.752263,0.643868,0.807407
7,XGBoost,GA,f1,20,0.246790,0.029577,0.242424,0.191176,0.296296
8,XGBoost,GA,mcc,20,0.253344,0.041367,0.262281,0.166674,0.322674
9,XGBoost,GA,pr_auc,20,0.198975,0.047566,0.174805,0.139881,0.272379


Deskriptif disimpan ke: ga_rs_analysis_output\ga_rs_descriptive_statistics.csv


In [4]:
# ------------------------------------------------------------
# 3. Paired Wilcoxon test: GA vs Random Search
#    Holm correction is applied within each model across all prediction metrics.
# ------------------------------------------------------------

results = []

for model in sorted(df["Model"].unique()):
    model_df = df[df["Model"] == model].copy()
    ga_df = model_df[model_df["Optimizer"] == "GA"]
    rs_df = model_df[model_df["Optimizer"] == "Random Search"]

    print("\n" + "=" * 90 + f"\nMODEL: {model}\n" + "=" * 90)

    for metric in metrics:
        ga_part = ga_df[key[2:] + [metric]].copy()  # Repeat, Outer_Fold, Split_ID, metric
        rs_part = rs_df[key[2:] + [metric]].copy()
        paired = ga_part.merge(
            rs_part,
            on=["Repeat", "Outer_Fold", "Split_ID"],
            how="inner",
            suffixes=("_GA", "_RS"),
            validate="one_to_one",
        ).dropna(subset=[f"{metric}_GA", f"{metric}_RS"])

        x = paired[f"{metric}_GA"].astype(float).to_numpy()
        y = paired[f"{metric}_RS"].astype(float).to_numpy()
        diff = x - y

        row = {
            "Model": model,
            "Metric": metric,
            "N_paired_splits": len(paired),
            "Mean_GA": np.mean(x) if len(x) else np.nan,
            "SD_GA": np.std(x, ddof=1) if len(x) > 1 else np.nan,
            "Mean_RandomSearch": np.mean(y) if len(y) else np.nan,
            "SD_RandomSearch": np.std(y, ddof=1) if len(y) > 1 else np.nan,
            "Mean_difference_GA_minus_RS": np.mean(diff) if len(diff) else np.nan,
            "Median_difference_GA_minus_RS": np.median(diff) if len(diff) else np.nan,
            "Wilcoxon_statistic": np.nan,
            "p_value_raw": np.nan,
            "Effect_size_rank_biserial": np.nan,
            "Status": "Not tested",
        }

        if len(paired) < 2:
            row["Status"] = "Insufficient paired observations"
        elif np.allclose(diff, 0):
            row["Status"] = "All paired differences are zero"
            row["p_value_raw"] = 1.0
            row["Wilcoxon_statistic"] = 0.0
            row["Effect_size_rank_biserial"] = 0.0
        else:
            try:
                stat, pval = wilcoxon(x, y, alternative="two-sided", zero_method="wilcox")
                row["Wilcoxon_statistic"] = float(stat)
                row["p_value_raw"] = float(pval)
                row["Status"] = "Test performed"

                # Rank-biserial correlation for the paired Wilcoxon test
                nonzero = diff[~np.isclose(diff, 0)]
                abs_diff = np.abs(nonzero)
                ranks = pd.Series(abs_diff).rank(method="average").to_numpy()
                w_pos = ranks[nonzero > 0].sum()
                w_neg = ranks[nonzero < 0].sum()
                denom = w_pos + w_neg
                row["Effect_size_rank_biserial"] = (
                    float((w_pos - w_neg) / denom) if denom else 0.0
                )
            except ValueError as exc:
                row["Status"] = f"Test error: {exc}"

        results.append(row)

    # Print a concise split-pairing summary for transparency
    print(f"Number of GA splits: {len(ga_df)} | Random Search: {len(rs_df)}")

stats = pd.DataFrame(results)

# Apply Holm correction within each model across all prediction metrics
stats["p_value_holm_within_model"] = np.nan
stats["Significant_Holm_0.05"] = False
for model, idx in stats.groupby("Model").groups.items():
    valid_idx = [i for i in idx if pd.notna(stats.loc[i, "p_value_raw"])]
    if valid_idx:
        reject, p_adj, _, _ = multipletests(
            stats.loc[valid_idx, "p_value_raw"].astype(float).to_numpy(),
            alpha=ALPHA,
            method="holm",
        )
        stats.loc[valid_idx, "p_value_holm_within_model"] = p_adj
        stats.loc[valid_idx, "Significant_Holm_0.05"] = reject

numeric_cols = stats.select_dtypes(include=[np.number]).columns
stats[numeric_cols] = stats[numeric_cols].round(6)

print("\nSTATISTICAL TEST RESULTS — reported separately for each model")
for model in sorted(stats["Model"].unique()):
    print("\n" + "=" * 90 + f"\nMODEL: {model}\n" + "=" * 90)
    display(stats[stats["Model"] == model].reset_index(drop=True))

stats_path = OUTPUT_DIR / "ga_rs_wilcoxon_holm_per_model.csv"
stats.to_csv(stats_path, index=False)
print("Statistical results saved to:", stats_path)


MODEL: Logistic Regression
Jumlah split GA: 20 | Random Search: 20

MODEL: Random Forest
Jumlah split GA: 20 | Random Search: 20

MODEL: SVM
Jumlah split GA: 20 | Random Search: 20

MODEL: XGBoost
Jumlah split GA: 20 | Random Search: 20

HASIL UJI STATISTIK — dipisahkan per model

MODEL: Logistic Regression


,Model,Metric,N_paired_splits,Mean_GA,SD_GA,Mean_RandomSearch,SD_RandomSearch,Mean_difference_GA_minus_RS,Median_difference_GA_minus_RS,Wilcoxon_statistic,p_value_raw,Effect_size_rank_biserial,Status,p_value_holm_within_model,Significant_Holm_0.05
0,Logistic Regression,accuracy,20,0.736204,0.013744,0.735910,0.013874,0.000294,0.000000,2.0,0.256839,0.600000,Test performed,1.000000,False
1,Logistic Regression,precision,20,0.133609,0.009893,0.133230,0.009632,0.000379,0.000000,2.0,0.138011,0.733333,Test performed,1.000000,False
2,Logistic Regression,recall_sensitivity,20,0.805333,0.081063,0.803333,0.080655,0.002000,0.000000,0.0,0.317311,1.000000,Test performed,1.000000,False
3,Logistic Regression,specificity,20,0.732673,0.016463,0.732467,0.016706,0.000206,0.000000,6.0,0.680280,0.200000,Test performed,1.000000,False
4,Logistic Regression,f1,20,0.229074,0.017201,0.228432,0.016767,0.000642,0.000000,1.0,0.079616,0.866667,Test performed,0.796158,False
5,Logistic Regression,roc_auc,20,0.839247,0.024611,0.838979,0.024767,0.000268,-0.000041,89.5,0.824659,0.057895,Test performed,1.000000,False
6,Logistic Regression,average_precision,20,0.212296,0.042173,0.212234,0.043174,0.000062,-0.000101,95.0,0.728506,-0.095238,Test performed,1.000000,False
7,Logistic Regression,pr_auc,20,0.199303,0.041982,0.199271,0.042897,0.000031,-0.000132,85.0,0.474905,-0.190476,Test performed,1.000000,False
8,Logistic Regression,balanced_accuracy,20,0.769003,0.036310,0.767900,0.035846,0.001103,0.000000,2.0,0.138011,0.733333,Test performed,1.000000,False
9,Logistic Regression,mcc,20,0.254317,0.032076,0.253248,0.031505,0.001069,0.000000,3.0,0.224916,0.600000,Test performed,1.000000,False



MODEL: Random Forest


,Model,Metric,N_paired_splits,Mean_GA,SD_GA,Mean_RandomSearch,SD_RandomSearch,Mean_difference_GA_minus_RS,Median_difference_GA_minus_RS,Wilcoxon_statistic,p_value_raw,Effect_size_rank_biserial,Status,p_value_holm_within_model,Significant_Holm_0.05
0,Random Forest,accuracy,20,0.787280,0.040051,0.797260,0.036767,-0.009980,-0.003914,61.5,0.176611,-0.352632,Test performed,1.0,False
1,Random Forest,precision,20,0.153827,0.025613,0.158766,0.024125,-0.004939,-0.000901,79.0,0.348810,-0.247619,Test performed,1.0,False
2,Random Forest,recall_sensitivity,20,0.724917,0.098274,0.715000,0.096023,0.009917,0.000000,32.5,0.609155,0.166667,Test performed,1.0,False
3,Random Forest,specificity,20,0.790486,0.044589,0.801494,0.040947,-0.011008,-0.003086,75.5,0.294252,-0.280952,Test performed,1.0,False
4,Random Forest,f1,20,0.252374,0.035818,0.258466,0.033640,-0.006092,-0.002150,79.0,0.348810,-0.247619,Test performed,1.0,False
5,Random Forest,roc_auc,20,0.834624,0.027313,0.834259,0.026886,0.000365,0.000123,95.0,0.728506,0.095238,Test performed,1.0,False
6,Random Forest,average_precision,20,0.199857,0.041165,0.198761,0.043565,0.001096,-0.001853,90.0,0.595819,0.142857,Test performed,1.0,False
7,Random Forest,pr_auc,20,0.186737,0.041150,0.185955,0.043600,0.000782,-0.001770,95.0,0.728506,0.095238,Test performed,1.0,False
8,Random Forest,balanced_accuracy,20,0.757701,0.041780,0.758247,0.041486,-0.000546,-0.001543,94.5,0.728506,-0.100000,Test performed,1.0,False
9,Random Forest,mcc,20,0.264573,0.044543,0.269245,0.042837,-0.004672,-0.001484,89.0,0.570597,-0.152381,Test performed,1.0,False



MODEL: SVM


,Model,Metric,N_paired_splits,Mean_GA,SD_GA,Mean_RandomSearch,SD_RandomSearch,Mean_difference_GA_minus_RS,Median_difference_GA_minus_RS,Wilcoxon_statistic,p_value_raw,Effect_size_rank_biserial,Status,p_value_holm_within_model,Significant_Holm_0.05
0,SVM,accuracy,20,0.714286,0.017013,0.708317,0.020715,0.005969,0.001957,10.0,0.012937,0.780220,Test performed,0.119974,False
1,SVM,precision,20,0.122693,0.010865,0.121037,0.011484,0.001655,0.000363,25.0,0.151956,0.450549,Test performed,1.000000,False
2,SVM,recall_sensitivity,20,0.789083,0.070162,0.793167,0.067853,-0.004083,0.000000,0.0,0.179712,-1.000000,Test performed,1.000000,False
3,SVM,specificity,20,0.710452,0.017934,0.703977,0.021902,0.006475,0.002058,7.0,0.011997,0.820513,Test performed,0.119974,False
4,SVM,f1,20,0.212287,0.018382,0.209911,0.019059,0.002376,0.000540,26.0,0.172955,0.428571,Test performed,1.000000,False
5,SVM,roc_auc,20,0.827937,0.024622,0.827711,0.022060,0.000226,-0.000123,84.5,0.672598,-0.110526,Test performed,1.000000,False
6,SVM,average_precision,20,0.200699,0.055533,0.197245,0.052083,0.003454,0.000142,78.0,0.493901,0.178947,Test performed,1.000000,False
7,SVM,pr_auc,20,0.187301,0.055409,0.183775,0.050537,0.003527,0.000138,85.0,0.687374,0.105263,Test performed,1.000000,False
8,SVM,balanced_accuracy,20,0.749768,0.035211,0.748572,0.034288,0.001196,0.000514,29.0,0.248719,0.362637,Test performed,1.000000,False
9,SVM,mcc,20,0.231968,0.032519,0.229777,0.032212,0.002191,0.000659,29.0,0.248864,0.362637,Test performed,1.000000,False



MODEL: XGBoost


,Model,Metric,N_paired_splits,Mean_GA,SD_GA,Mean_RandomSearch,SD_RandomSearch,Mean_difference_GA_minus_RS,Median_difference_GA_minus_RS,Wilcoxon_statistic,p_value_raw,Effect_size_rank_biserial,Status,p_value_holm_within_model,Significant_Holm_0.05
0,XGBoost,accuracy,20,0.793346,0.033527,0.812622,0.037637,-0.019276,-0.013699,53.5,0.094809,-0.436842,Test performed,0.853281,False
1,XGBoost,precision,20,0.151058,0.020032,0.161043,0.024434,-0.009986,-0.007183,60.0,0.097307,-0.428571,Test performed,0.853281,False
2,XGBoost,recall_sensitivity,20,0.690250,0.103613,0.662500,0.139619,0.027750,0.040000,72.0,0.353497,0.242105,Test performed,1.000000,False
3,XGBoost,specificity,20,0.798599,0.038290,0.820307,0.044254,-0.021708,-0.017490,62.0,0.113987,-0.409524,Test performed,0.853281,False
4,XGBoost,f1,20,0.246790,0.029577,0.256762,0.037206,-0.009971,-0.005532,68.0,0.176853,-0.352381,Test performed,1.000000,False
5,XGBoost,roc_auc,20,0.837466,0.029109,0.842335,0.027144,-0.004869,-0.006132,44.0,0.021484,-0.580952,Test performed,0.214844,False
6,XGBoost,average_precision,20,0.211675,0.049010,0.217243,0.054902,-0.005567,-0.005785,77.0,0.311794,-0.266667,Test performed,1.000000,False
7,XGBoost,pr_auc,20,0.198975,0.047566,0.203676,0.053981,-0.004701,-0.006692,72.0,0.230513,-0.314286,Test performed,1.000000,False
8,XGBoost,balanced_accuracy,20,0.744424,0.042708,0.741403,0.056682,0.003021,-0.007716,101.0,0.898317,-0.038095,Test performed,1.000000,False
9,XGBoost,mcc,20,0.253344,0.041367,0.259633,0.052958,-0.006289,-0.016762,75.0,0.277355,-0.285714,Test performed,1.000000,False


Hasil statistik disimpan ke: ga_rs_analysis_output\ga_rs_wilcoxon_holm_per_model.csv


In [5]:
# ------------------------------------------------------------
# 4. Performance difference summary: GA - Random Search
# ------------------------------------------------------------

# A positive difference means a higher score for GA. For time/cost metrics,
# the direction has a different interpretation, so they are not included in performance testing.
summary_cols = [
    "Model", "Metric", "N_paired_splits", "Mean_GA", "Mean_RandomSearch",
    "Mean_difference_GA_minus_RS", "Median_difference_GA_minus_RS",
    "p_value_raw", "p_value_holm_within_model", "Effect_size_rank_biserial",
    "Significant_Holm_0.05", "Status"
]
summary_cols = [c for c in summary_cols if c in stats.columns]
summary = stats[summary_cols].copy()
summary_path = OUTPUT_DIR / "ga_rs_performance_comparison_summary.csv"
summary.to_csv(summary_path, index=False)
display(summary)

print("\nCompleted. Output files:")
for path in [clean_path, desc_path, stats_path, summary_path]:
    print("-", path.resolve())

,Model,Metric,N_paired_splits,Mean_GA,Mean_RandomSearch,Mean_difference_GA_minus_RS,Median_difference_GA_minus_RS,p_value_raw,p_value_holm_within_model,Effect_size_rank_biserial,Significant_Holm_0.05,Status
0,Logistic Regression,accuracy,20,0.736204,0.735910,0.000294,0.000000,0.256839,1.000000,0.600000,False,Test performed
1,Logistic Regression,precision,20,0.133609,0.133230,0.000379,0.000000,0.138011,1.000000,0.733333,False,Test performed
2,Logistic Regression,recall_sensitivity,20,0.805333,0.803333,0.002000,0.000000,0.317311,1.000000,1.000000,False,Test performed
3,Logistic Regression,specificity,20,0.732673,0.732467,0.000206,0.000000,0.680280,1.000000,0.200000,False,Test performed
4,Logistic Regression,f1,20,0.229074,0.228432,0.000642,0.000000,0.079616,0.796158,0.866667,False,Test performed
5,Logistic Regression,roc_auc,20,0.839247,0.838979,0.000268,-0.000041,0.824659,1.000000,0.057895,False,Test performed
6,Logistic Regression,average_precision,20,0.212296,0.212234,0.000062,-0.000101,0.728506,1.000000,-0.095238,False,Test performed
7,Logistic Regression,pr_auc,20,0.199303,0.199271,0.000031,-0.000132,0.474905,1.000000,-0.190476,False,Test performed
8,Logistic Regression,balanced_accuracy,20,0.769003,0.767900,0.001103,0.000000,0.138011,1.000000,0.733333,False,Test performed
9,Logistic Regression,mcc,20,0.254317,0.253248,0.001069,0.000000,0.224916,1.000000,0.600000,False,Test performed



Selesai. File output:
- D:\KULIAH\SEMESTER 7\SKRIPSI\jurnal pa irwan\codingan\GA vs RS 9\ga_rs_analysis_output\ga_rs_fold_results_clean.csv
- D:\KULIAH\SEMESTER 7\SKRIPSI\jurnal pa irwan\codingan\GA vs RS 9\ga_rs_analysis_output\ga_rs_descriptive_statistics.csv
- D:\KULIAH\SEMESTER 7\SKRIPSI\jurnal pa irwan\codingan\GA vs RS 9\ga_rs_analysis_output\ga_rs_wilcoxon_holm_per_model.csv
- D:\KULIAH\SEMESTER 7\SKRIPSI\jurnal pa irwan\codingan\GA vs RS 9\ga_rs_analysis_output\ga_rs_performance_comparison_summary.csv
